# Notebook 03 — Per-Product Time-Series Analysis

**Explorer 03 | R5 EDA Swarm**

Lenses: ADF stationarity test, AR(1)/AR(5) on returns, trend slope + R², FFT top-5 frequencies.
Covers all 50 products across days 2, 3, 4.

Triage rubric (from AGENT_BRIEF.md):
- **likely exploitable**: AR(1) |ρ| > 0.3
- **probably tradable**: AR(1) |ρ| between 0.1–0.3
- **probably noise**: AR(1) |ρ| < 0.1


## Cell 1 — Imports and data loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal as scipy_signal
from scipy import stats as scipy_stats
from statsmodels.tsa.stattools import adfuller, acf
import warnings
warnings.filterwarnings('ignore')

BASE = '/Users/bensinek/Documents/Coding/Prosperity4/data/round_5/prices'
DAYS = [2, 3, 4]

frames = []
for d in DAYS:
    df = pd.read_csv(f'{BASE}/prices_round_5_day_{d}.csv', sep=';')
    frames.append(df)
raw = pd.concat(frames, ignore_index=True)

# Use mid_price as the price series
raw = raw[['day', 'timestamp', 'product', 'mid_price']].copy()
raw = raw.sort_values(['product', 'day', 'timestamp']).reset_index(drop=True)

PRODUCTS = sorted(raw['product'].unique())
CATEGORIES = {
    'GALAXY_SOUNDS': [p for p in PRODUCTS if p.startswith('GALAXY_SOUNDS')],
    'SLEEP_POD':     [p for p in PRODUCTS if p.startswith('SLEEP_POD')],
    'MICROCHIP':     [p for p in PRODUCTS if p.startswith('MICROCHIP')],
    'PEBBLES':       [p for p in PRODUCTS if p.startswith('PEBBLES')],
    'ROBOT':         [p for p in PRODUCTS if p.startswith('ROBOT')],
    'UV_VISOR':      [p for p in PRODUCTS if p.startswith('UV_VISOR')],
    'TRANSLATOR':    [p for p in PRODUCTS if p.startswith('TRANSLATOR')],
    'PANEL':         [p for p in PRODUCTS if p.startswith('PANEL')],
    'OXYGEN_SHAKE':  [p for p in PRODUCTS if p.startswith('OXYGEN_SHAKE')],
    'SNACKPACK':     [p for p in PRODUCTS if p.startswith('SNACKPACK')],
}

print(f'Total rows: {len(raw):,}')
print(f'Products: {len(PRODUCTS)}')
print(f'Days: {raw["day"].unique()}')
print(f'Ticks per product per day: {raw.groupby(["product","day"]).size().unique()}')


Total rows: 1,500,000
Products: 50
Days: [2 3 4]
Ticks per product per day: [10000]


## Cell 2 — ADF stationarity test (levels and returns) per product per day

ADF null hypothesis: the series has a unit root (non-stationary).  
- p < 0.05 → reject unit root → stationary at 5% level.  
- We test both **levels** (mid_price) and **returns** (log differences).  
- AR(1) on non-stationary levels is spurious; we track both to catch which products have stationary price levels (mean-reverting) vs. non-stationary (random walk).


In [2]:
def run_adf(series, max_lags=20):
    """Run ADF test, return (statistic, p_value, lags_used)."""
    try:
        result = adfuller(series.dropna(), maxlag=max_lags, autolag='AIC')
        return result[0], result[1], result[2]
    except Exception as e:
        return np.nan, np.nan, np.nan

adf_rows = []
for prod in PRODUCTS:
    for day in DAYS:
        sub = raw[(raw['product'] == prod) & (raw['day'] == day)].sort_values('timestamp')
        levels = sub['mid_price'].values
        returns = np.diff(np.log(levels + 1e-9))
        
        stat_lev, p_lev, lag_lev = run_adf(pd.Series(levels))
        stat_ret, p_ret, lag_ret = run_adf(pd.Series(returns))
        
        adf_rows.append({
            'product': prod,
            'day': day,
            'adf_stat_levels': stat_lev,
            'adf_p_levels': p_lev,
            'adf_lags_levels': lag_lev,
            'stationary_levels': p_lev < 0.05 if not np.isnan(p_lev) else None,
            'adf_stat_returns': stat_ret,
            'adf_p_returns': p_ret,
            'adf_lags_returns': lag_ret,
            'stationary_returns': p_ret < 0.05 if not np.isnan(p_ret) else None,
        })

adf_df = pd.DataFrame(adf_rows)

# Summary: fraction of days each product has stationary levels
adf_summary = adf_df.groupby('product').agg(
    frac_stationary_levels=('stationary_levels', 'mean'),
    frac_stationary_returns=('stationary_returns', 'mean'),
    mean_adf_p_levels=('adf_p_levels', 'mean'),
    mean_adf_p_returns=('adf_p_returns', 'mean'),
).reset_index()

adf_summary = adf_summary.sort_values('frac_stationary_levels', ascending=False)
print("=== ADF Summary (fraction of 3 days where stationary at p<0.05) ===")
print(adf_summary.to_string(index=False))


=== ADF Summary (fraction of 3 days where stationary at p<0.05) ===
                      product  frac_stationary_levels  frac_stationary_returns  mean_adf_p_levels  mean_adf_p_returns
                 UV_VISOR_RED                0.666667                      1.0           0.195365                 0.0
             MICROCHIP_CIRCLE                0.333333                      1.0           0.449830                 0.0
               UV_VISOR_AMBER                0.333333                      1.0           0.230690                 0.0
  OXYGEN_SHAKE_EVENING_BREATH                0.333333                      1.0           0.366396                 0.0
       TRANSLATOR_ASTRO_BLACK                0.333333                      1.0           0.176058                 0.0
         SNACKPACK_STRAWBERRY                0.333333                      1.0           0.169637                 0.0
    GALAXY_SOUNDS_BLACK_HOLES                0.000000                      1.0           0.843583         

## Cell 3 — AR(1) and AR(5) on returns

We regress returns on lagged returns using OLS.  
- AR(1): r_t = c + ρ·r_{t-1} + ε  
- AR(5): r_t = c + Σ_{k=1}^{5} ρ_k·r_{t-k} + ε  

ρ is estimated per (product, day). We use returns (not levels) to avoid spurious regression on non-stationary series.  
**Note**: even if levels are stationary (mean-reverting), AR(1) on RETURNS captures short-horizon predictability, which is the trading signal.


In [3]:
def fit_ar(returns, p=1):
    """Fit AR(p) via OLS on returns array. Returns (rho_1, ..., rho_p, r2, pvalue_rho1)."""
    r = np.array(returns)
    r = r[np.isfinite(r)]
    if len(r) < p + 10:
        return [np.nan]*p + [np.nan, np.nan]
    
    # Build design matrix
    n = len(r) - p
    Y = r[p:]
    X = np.column_stack([r[p-k-1:n+p-k-1] for k in range(p)])
    X = np.column_stack([np.ones(n), X])
    
    try:
        result = np.linalg.lstsq(X, Y, rcond=None)
        coeffs = result[0]
        Y_hat = X @ coeffs
        ss_res = np.sum((Y - Y_hat)**2)
        ss_tot = np.sum((Y - Y.mean())**2)
        r2 = 1 - ss_res/ss_tot if ss_tot > 0 else 0.0
        
        # p-value for rho_1 via t-statistic
        rho_1 = coeffs[1]
        # residual variance
        dof = n - (p + 1)
        if dof > 0:
            sigma2 = ss_res / dof
            XtXinv = np.linalg.pinv(X.T @ X)
            se_rho1 = np.sqrt(sigma2 * XtXinv[1,1])
            t_stat = rho_1 / se_rho1 if se_rho1 > 0 else np.nan
            pval = 2 * (1 - scipy_stats.t.cdf(abs(t_stat), df=dof)) if not np.isnan(t_stat) else np.nan
        else:
            pval = np.nan
        
        rhos = list(coeffs[1:p+1])
        return rhos + [r2, pval]
    except:
        return [np.nan]*p + [np.nan, np.nan]

ar_rows = []
for prod in PRODUCTS:
    for day in DAYS:
        sub = raw[(raw['product'] == prod) & (raw['day'] == day)].sort_values('timestamp')
        levels = sub['mid_price'].values
        returns = np.diff(np.log(levels + 1e-9))
        
        ar1_res = fit_ar(returns, p=1)
        ar5_res = fit_ar(returns, p=5)
        
        row = {
            'product': prod,
            'day': day,
            'ar1_rho1': ar1_res[0],
            'ar1_r2': ar1_res[1],
            'ar1_pval': ar1_res[2],
        }
        for k in range(5):
            row[f'ar5_rho{k+1}'] = ar5_res[k]
        row['ar5_r2'] = ar5_res[5]
        row['ar5_pval_rho1'] = ar5_res[6]
        ar_rows.append(row)

ar_df = pd.DataFrame(ar_rows)

# Summary across days
ar_summary = ar_df.groupby('product').agg(
    mean_ar1_rho1=('ar1_rho1', 'mean'),
    std_ar1_rho1=('ar1_rho1', 'std'),
    mean_ar1_r2=('ar1_r2', 'mean'),
    mean_ar1_pval=('ar1_pval', 'mean'),
    mean_ar5_r2=('ar5_r2', 'mean'),
).reset_index()
ar_summary['abs_rho1'] = ar_summary['mean_ar1_rho1'].abs()
ar_summary = ar_summary.sort_values('abs_rho1', ascending=False)

print("=== AR(1) summary — sorted by |ρ₁| (mean across days 2/3/4) ===")
print(ar_summary[['product','mean_ar1_rho1','std_ar1_rho1','abs_rho1','mean_ar1_r2','mean_ar1_pval','mean_ar5_r2']].to_string(index=False))


=== AR(1) summary — sorted by |ρ₁| (mean across days 2/3/4) ===
                      product  mean_ar1_rho1  std_ar1_rho1  abs_rho1  mean_ar1_r2  mean_ar1_pval  mean_ar5_r2
                ROBOT_IRONING      -0.116754      0.037763  0.116754     0.014578   2.960595e-16     0.016050
  OXYGEN_SHAKE_EVENING_BREATH      -0.111778      0.045233  0.111778     0.013859   2.960595e-15     0.014469
                 ROBOT_DISHES      -0.097614      0.166104  0.097614     0.027922   5.596703e-01     0.039824
       OXYGEN_SHAKE_CHOCOLATE      -0.075968      0.059682  0.075968     0.008146   1.451581e-01     0.009112
          SNACKPACK_CHOCOLATE      -0.030843      0.006248  0.030843     0.000977   5.848437e-03     0.001423
            SNACKPACK_VANILLA      -0.026973      0.004214  0.026973     0.000739   1.064932e-02     0.001149
          SNACKPACK_PISTACHIO      -0.025087      0.007823  0.025087     0.000670   2.797592e-02     0.000932
             MICROCHIP_SQUARE      -0.022137      0.0114

## Cell 4 — Triage bucketing by AR(1) rubric

In [4]:
def triage_ar1(rho):
    if abs(rho) > 0.3:
        return 'likely_exploitable'
    elif abs(rho) >= 0.1:
        return 'probably_tradable'
    else:
        return 'probably_noise'

ar_summary['triage_ar1'] = ar_summary['mean_ar1_rho1'].apply(triage_ar1)
ar_summary['direction'] = ar_summary['mean_ar1_rho1'].apply(lambda r: 'momentum' if r > 0 else 'mean_revert')

print("\n=== LIKELY EXPLOITABLE (|ρ₁| > 0.3) ===")
likely = ar_summary[ar_summary['triage_ar1'] == 'likely_exploitable']
print(likely[['product','mean_ar1_rho1','std_ar1_rho1','mean_ar1_pval','direction']].to_string(index=False))

print(f"\nCount: {len(likely)}")

print("\n=== PROBABLY TRADABLE (0.1 ≤ |ρ₁| ≤ 0.3) ===")
tradable = ar_summary[ar_summary['triage_ar1'] == 'probably_tradable']
print(tradable[['product','mean_ar1_rho1','std_ar1_rho1','mean_ar1_pval','direction']].to_string(index=False))
print(f"\nCount: {len(tradable)}")

print("\n=== PROBABLY NOISE (|ρ₁| < 0.1) ===")
noise = ar_summary[ar_summary['triage_ar1'] == 'probably_noise']
print(noise[['product','mean_ar1_rho1','std_ar1_rho1','mean_ar1_pval','direction']].to_string(index=False))
print(f"\nCount: {len(noise)}")



=== LIKELY EXPLOITABLE (|ρ₁| > 0.3) ===
Empty DataFrame
Columns: [product, mean_ar1_rho1, std_ar1_rho1, mean_ar1_pval, direction]
Index: []

Count: 0

=== PROBABLY TRADABLE (0.1 ≤ |ρ₁| ≤ 0.3) ===
                    product  mean_ar1_rho1  std_ar1_rho1  mean_ar1_pval   direction
              ROBOT_IRONING      -0.116754      0.037763   2.960595e-16 mean_revert
OXYGEN_SHAKE_EVENING_BREATH      -0.111778      0.045233   2.960595e-15 mean_revert

Count: 2

=== PROBABLY NOISE (|ρ₁| < 0.1) ===
                      product  mean_ar1_rho1  std_ar1_rho1  mean_ar1_pval   direction
                 ROBOT_DISHES      -0.097614      0.166104       0.559670 mean_revert
       OXYGEN_SHAKE_CHOCOLATE      -0.075968      0.059682       0.145158 mean_revert
          SNACKPACK_CHOCOLATE      -0.030843      0.006248       0.005848 mean_revert
            SNACKPACK_VANILLA      -0.026973      0.004214       0.010649 mean_revert
          SNACKPACK_PISTACHIO      -0.025087      0.007823       0.027976 

## Cell 5 — Per-day AR(1) breakdown (stability check)

A product is only truly exploitable if the AR(1) signal is stable across all 3 days. Here we print the per-day ρ₁ for the likely-exploitable and probably-tradable buckets.


In [5]:
focus_products = list(likely['product']) + list(tradable['product'])
per_day = ar_df[ar_df['product'].isin(focus_products)][['product','day','ar1_rho1','ar1_r2','ar1_pval']].copy()
per_day_pivot = per_day.pivot_table(index='product', columns='day', values=['ar1_rho1','ar1_r2','ar1_pval']).round(5)
per_day_pivot.columns = [f'{v}_d{d}' for v,d in per_day_pivot.columns]
per_day_pivot = per_day_pivot.reset_index()

# Sort by mean |rho| descending
per_day_pivot['mean_abs_rho'] = per_day_pivot[['ar1_rho1_d2','ar1_rho1_d3','ar1_rho1_d4']].abs().mean(axis=1)
per_day_pivot = per_day_pivot.sort_values('mean_abs_rho', ascending=False)

print("=== Per-day ρ₁ for likely-exploitable + probably-tradable products ===")
rho_cols = ['product','ar1_rho1_d2','ar1_rho1_d3','ar1_rho1_d4','mean_abs_rho']
print(per_day_pivot[rho_cols].to_string(index=False))

print("\n=== Stability flag: all 3 days same sign AND |ρ₁| consistent ===")
for _, row in per_day_pivot.iterrows():
    d2, d3, d4 = row['ar1_rho1_d2'], row['ar1_rho1_d3'], row['ar1_rho1_d4']
    vals = [d2, d3, d4]
    same_sign = all(v > 0 for v in vals) or all(v < 0 for v in vals)
    rng = max(vals) - min(vals)
    print(f"  {row['product']:45s} same_sign={same_sign}  range={rng:.5f}")


=== Per-day ρ₁ for likely-exploitable + probably-tradable products ===
                    product  ar1_rho1_d2  ar1_rho1_d3  ar1_rho1_d4  mean_abs_rho
              ROBOT_IRONING     -0.15559     -0.08017     -0.11450      0.116753
OXYGEN_SHAKE_EVENING_BREATH     -0.16303     -0.09486     -0.07745      0.111780

=== Stability flag: all 3 days same sign AND |ρ₁| consistent ===
  ROBOT_IRONING                                 same_sign=True  range=0.07542
  OXYGEN_SHAKE_EVENING_BREATH                   same_sign=True  range=0.08558


## Cell 6 — Trend slope and R² per product per day

Linear OLS fit of mid_price vs. timestamp.  
- Slope = average price change per tick.  
- R² = fraction of price variance explained by linear time trend.  
- High R² means persistent directional drift — "probably tradable" by trend-following; could be irrelevant after R² correction.


In [6]:
trend_rows = []
for prod in PRODUCTS:
    for day in DAYS:
        sub = raw[(raw['product'] == prod) & (raw['day'] == day)].sort_values('timestamp')
        ts = sub['timestamp'].values.astype(float)
        price = sub['mid_price'].values
        
        if len(ts) < 5:
            trend_rows.append({'product': prod, 'day': day, 'slope': np.nan, 'intercept': np.nan, 'r2': np.nan, 'pval_slope': np.nan})
            continue
        
        slope, intercept, r_value, p_value, std_err = scipy_stats.linregress(ts, price)
        trend_rows.append({
            'product': prod,
            'day': day,
            'slope': slope,
            'intercept': intercept,
            'r2': r_value**2,
            'pval_slope': p_value,
        })

trend_df = pd.DataFrame(trend_rows)

trend_summary = trend_df.groupby('product').agg(
    mean_slope=('slope', 'mean'),
    mean_r2=('r2', 'mean'),
    mean_pval=('pval_slope', 'mean'),
    std_slope=('slope', 'std'),
).reset_index()
trend_summary = trend_summary.sort_values('mean_r2', ascending=False)

print("=== Trend R² summary (sorted by mean R² across days) ===")
print(trend_summary.to_string(index=False))

print(f"\nProducts with mean trend R² > 0.3 (per rubric): {(trend_summary['mean_r2'] > 0.3).sum()}")
high_trend = trend_summary[trend_summary['mean_r2'] > 0.3]
if len(high_trend) > 0:
    print(high_trend[['product','mean_slope','mean_r2','mean_pval']].to_string(index=False))


=== Trend R² summary (sorted by mean R² across days) ===
                      product  mean_slope  mean_r2     mean_pval  std_slope
                    PANEL_1X2    0.000275 0.766259  0.000000e+00   0.001363
              SLEEP_POD_NYLON    0.000394 0.737322  0.000000e+00   0.001054
                   PEBBLES_XS   -0.001760 0.704839  0.000000e+00   0.000525
        TRANSLATOR_SPACE_GRAY   -0.000374 0.660375  0.000000e+00   0.001285
     TRANSLATOR_GRAPHITE_MIST   -0.000420 0.656437  0.000000e+00   0.001361
             SLEEP_POD_COTTON    0.000368 0.645952  0.000000e+00   0.001619
          SLEEP_POD_POLYESTER    0.000647 0.638556  0.000000e+00   0.001338
             MICROCHIP_SQUARE    0.001288 0.635132  0.000000e+00   0.002325
                    PANEL_2X4    0.001115 0.621566  0.000000e+00   0.000285
              UV_VISOR_YELLOW    0.000038 0.609642 2.821789e-232   0.002303
                   PEBBLES_XL    0.002842 0.599814 4.656334e-223   0.003044
    GALAXY_SOUNDS_DARK_MATTER  

## Cell 7 — FFT: top-5 dominant frequencies per product

We compute the FFT of the **de-trended** mid_price series (detrended to avoid DC component dominating).  
For each product we report the top-5 frequencies by amplitude, normalized as cycles per 10,000 ticks.  
A sharp FFT peak >> median amplitude suggests embedded periodicity.  
We compute a peak-to-noise ratio: peak amplitude / median amplitude of all frequencies.


In [7]:
fft_rows = []
for prod in PRODUCTS:
    for day in DAYS:
        sub = raw[(raw['product'] == prod) & (raw['day'] == day)].sort_values('timestamp')
        price = sub['mid_price'].values
        
        if len(price) < 100:
            continue
        
        # Detrend
        price_dt = scipy_signal.detrend(price)
        
        N = len(price_dt)
        fft_vals = np.fft.rfft(price_dt)
        freqs = np.fft.rfftfreq(N)  # cycles per tick
        amplitudes = np.abs(fft_vals)
        
        # Exclude DC (freq=0)
        freqs_nondc = freqs[1:]
        amps_nondc = amplitudes[1:]
        
        # Top 5
        top5_idx = np.argsort(amps_nondc)[::-1][:5]
        top5_freqs = freqs_nondc[top5_idx]
        top5_amps = amps_nondc[top5_idx]
        
        # Peak-to-noise: max amp / median amp
        pnr = top5_amps[0] / (np.median(amps_nondc) + 1e-12)
        
        # Period in ticks for top freq
        period_top1 = 1.0 / top5_freqs[0] if top5_freqs[0] > 0 else np.nan
        
        row = {
            'product': prod,
            'day': day,
            'peak_to_noise_ratio': pnr,
            'top1_freq_per_tick': top5_freqs[0],
            'top1_period_ticks': period_top1,
            'top1_amp': top5_amps[0],
        }
        for k in range(5):
            row[f'top{k+1}_freq'] = top5_freqs[k] if k < len(top5_freqs) else np.nan
            row[f'top{k+1}_amp'] = top5_amps[k] if k < len(top5_amps) else np.nan
        fft_rows.append(row)

fft_df = pd.DataFrame(fft_rows)

fft_summary = fft_df.groupby('product').agg(
    mean_pnr=('peak_to_noise_ratio', 'mean'),
    mean_top1_period=('top1_period_ticks', 'mean'),
    std_pnr=('peak_to_noise_ratio', 'std'),
).reset_index()
fft_summary = fft_summary.sort_values('mean_pnr', ascending=False)

print("=== FFT peak-to-noise ratio (higher = more periodic signal) ===")
print(fft_summary.to_string(index=False))

HIGH_PNR_THRESH = 10.0  # arbitrary initial threshold
high_pnr = fft_summary[fft_summary['mean_pnr'] > HIGH_PNR_THRESH]
print(f"\nProducts with mean FFT peak-to-noise ratio > {HIGH_PNR_THRESH}: {len(high_pnr)}")
if len(high_pnr) > 0:
    print(high_pnr.to_string(index=False))


=== FFT peak-to-noise ratio (higher = more periodic signal) ===
                      product    mean_pnr  mean_top1_period     std_pnr
              UV_VISOR_ORANGE 2716.566172      10000.000000 1586.443535
                    PANEL_1X4 2541.099694       8333.333333  416.235940
   GALAXY_SOUNDS_SOLAR_FLAMES 2280.774643       8333.333333  647.943697
                    PANEL_2X2 2272.791082       8333.333333 1242.936818
              SLEEP_POD_SUEDE 2169.436341       8333.333333 1222.636171
            OXYGEN_SHAKE_MINT 2093.320130      10000.000000  982.378423
                    PEBBLES_L 1989.470255       8333.333333  150.171265
                ROBOT_IRONING 1913.718685       8333.333333 1050.026797
GALAXY_SOUNDS_PLANETARY_RINGS 1868.021524      10000.000000 1263.109789
         TRANSLATOR_VOID_BLUE 1831.530066       8333.333333  483.844000
          OXYGEN_SHAKE_GARLIC 1830.380997       7777.777778 1020.301274
                    PANEL_4X4 1744.868384       8333.333333  367.272832


## Cell 8 — FFT spectra for top candidates (plot)

Plotting the FFT power spectrum for the top-10 products by peak-to-noise ratio, across all 3 days.  
This lets us visually confirm whether the spectral peak is narrow and consistent (true periodicity) or broad (noise artifact).


In [8]:
top10_fft_products = fft_summary.head(10)['product'].tolist()

fig, axes = plt.subplots(10, 3, figsize=(18, 30))
fig.suptitle('FFT Power Spectra — Top-10 by Peak-to-Noise Ratio (Days 2/3/4)', fontsize=14, y=1.01)

for i, prod in enumerate(top10_fft_products):
    for j, day in enumerate(DAYS):
        ax = axes[i][j]
        sub = raw[(raw['product'] == prod) & (raw['day'] == day)].sort_values('timestamp')
        price = sub['mid_price'].values
        
        if len(price) < 100:
            ax.set_visible(False)
            continue
        
        price_dt = scipy_signal.detrend(price)
        N = len(price_dt)
        fft_vals = np.fft.rfft(price_dt)
        freqs = np.fft.rfftfreq(N)
        amps = np.abs(fft_vals)
        
        # Plot only up to freq = 0.05 (long-period structure)
        mask = (freqs > 0) & (freqs <= 0.05)
        ax.plot(freqs[mask] * 10000, amps[mask], linewidth=0.7, color='steelblue')
        
        # Safe PNR lookup
        pnr_row = fft_df[(fft_df['product'] == prod) & (fft_df['day'] == day)]
        pnr_str = f"PNR={pnr_row['peak_to_noise_ratio'].values[0]:.1f}" if len(pnr_row) > 0 else "PNR=N/A"
        ax.set_title(f'{prod}\nDay {day} | {pnr_str}', fontsize=7)
        ax.set_xlabel('Freq (cycles/10k ticks)', fontsize=6)
        ax.set_ylabel('Amplitude', fontsize=6)
        ax.tick_params(labelsize=6)

plt.tight_layout()
plt.savefig('/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots/03_fft_top10.png', dpi=100, bbox_inches='tight')
plt.show()
print("FFT spectra plot saved.")


FFT spectra plot saved.


## Cell 9 — AR(1) heatmap: ρ₁ per product × day

Heatmap to visually confirm which products have stable AR(1) structure across all 3 days.


In [9]:
ar_pivot = ar_df.pivot_table(index='product', columns='day', values='ar1_rho1')
ar_pivot.columns = [f'Day {c}' for c in ar_pivot.columns]
# Sort by mean absolute rho
ar_pivot['mean_abs'] = ar_pivot.abs().mean(axis=1)
ar_pivot = ar_pivot.sort_values('mean_abs', ascending=False).drop(columns='mean_abs')

fig, ax = plt.subplots(figsize=(8, 16))
im = ax.imshow(ar_pivot.values, aspect='auto', cmap='RdBu_r', vmin=-0.5, vmax=0.5)
plt.colorbar(im, ax=ax, label='AR(1) ρ₁ on returns')
ax.set_xticks(range(len(ar_pivot.columns)))
ax.set_xticklabels(ar_pivot.columns)
ax.set_yticks(range(len(ar_pivot.index)))
ax.set_yticklabels(ar_pivot.index, fontsize=7)
ax.set_title('AR(1) ρ₁ Heatmap — all 50 products × 3 days\n(red=momentum, blue=mean-reversion)', fontsize=10)

# Add threshold lines
ax.axhline(-0.5, color='gray', lw=0.3)
plt.tight_layout()
plt.savefig('/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots/03_ar1_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()
print("AR(1) heatmap saved.")


AR(1) heatmap saved.


## Cell 10 — Price level time-series plots for all 50 products

Overview plots of mid_price over time for all products, grouped by category. 3 days overlaid.  
Useful to visually identify mean-reversion, trend, oscillation, or hardcoded-FV patterns.


In [10]:
colors = {2: 'steelblue', 3: 'darkorange', 4: 'seagreen'}

for cat_name, cat_products in CATEGORIES.items():
    fig, axes = plt.subplots(1, 5, figsize=(22, 4), sharey=False)
    fig.suptitle(f'Category: {cat_name} — Mid-price timeseries (Days 2/3/4)', fontsize=11)
    
    for ax, prod in zip(axes, cat_products):
        short = prod.replace(cat_name + '_', '')
        for day in DAYS:
            sub = raw[(raw['product'] == prod) & (raw['day'] == day)].sort_values('timestamp')
            ax.plot(sub['timestamp'].values, sub['mid_price'].values, 
                    color=colors[day], alpha=0.7, linewidth=0.6, label=f'Day {day}')
        
        # Annotate with AR(1) rho
        rho_val = ar_summary[ar_summary['product'] == prod]['mean_ar1_rho1'].values
        rho_str = f"ρ₁={rho_val[0]:.3f}" if len(rho_val) > 0 else ""
        ax.set_title(f'{short}\n{rho_str}', fontsize=8)
        ax.tick_params(labelsize=6)
        ax.set_xlabel('Tick', fontsize=6)
        if ax == axes[0]:
            ax.set_ylabel('Mid price', fontsize=7)
            ax.legend(fontsize=6)
    
    plt.tight_layout()
    safe_name = cat_name.replace(' ', '_').lower()
    plt.savefig(f'/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots/03_ts_{safe_name}.png', dpi=80, bbox_inches='tight')
    plt.show()
    print(f"Saved plot for {cat_name}")


Saved plot for GALAXY_SOUNDS


Saved plot for SLEEP_POD


Saved plot for MICROCHIP


Saved plot for PEBBLES


Saved plot for ROBOT


Saved plot for UV_VISOR


Saved plot for TRANSLATOR


Saved plot for PANEL


Saved plot for OXYGEN_SHAKE


Saved plot for SNACKPACK


## Cell 11 — Combined triage table (all 4 lenses)

Merge ADF stationarity, AR(1), trend R², and FFT peak-to-noise into one table.  
Final triage call per product from THIS notebook's lenses only (synthesis agent makes the final call).


In [11]:
combined = ar_summary[['product','mean_ar1_rho1','abs_rho1','mean_ar1_r2','triage_ar1','direction']].copy()
combined = combined.merge(
    adf_summary[['product','frac_stationary_levels','frac_stationary_returns','mean_adf_p_levels']],
    on='product', how='left'
)
combined = combined.merge(
    trend_summary[['product','mean_slope','mean_r2']].rename(columns={'mean_r2':'trend_r2'}),
    on='product', how='left'
)
combined = combined.merge(
    fft_summary[['product','mean_pnr','mean_top1_period']],
    on='product', how='left'
)

# Assign category
def get_cat(prod):
    for cat in CATEGORIES:
        if prod in CATEGORIES[cat]:
            return cat
    return 'UNKNOWN'
combined['category'] = combined['product'].apply(get_cat)

# Override triage if trend R² > 0.3 and AR(1) says noise
def final_triage(row):
    if row['triage_ar1'] == 'likely_exploitable':
        return 'likely_exploitable'
    if row['trend_r2'] > 0.3:
        return 'probably_tradable'
    return row['triage_ar1']

combined['triage_03'] = combined.apply(final_triage, axis=1)
combined = combined.sort_values(['triage_03','abs_rho1'], ascending=[True, False])

print("=== COMBINED TRIAGE TABLE (Notebook 03 lenses) ===")
cols = ['product','category','mean_ar1_rho1','abs_rho1','trend_r2','frac_stationary_levels',
        'mean_pnr','triage_03','direction']
print(combined[cols].to_string(index=False))

print("\n=== TRIAGE COUNTS ===")
print(combined['triage_03'].value_counts())


=== COMBINED TRIAGE TABLE (Notebook 03 lenses) ===
                      product      category  mean_ar1_rho1  abs_rho1  trend_r2  frac_stationary_levels    mean_pnr         triage_03   direction
          SNACKPACK_CHOCOLATE     SNACKPACK      -0.030843  0.030843  0.209775                0.000000 1366.163707    probably_noise mean_revert
            SNACKPACK_VANILLA     SNACKPACK      -0.026973  0.026973  0.267610                0.000000 1261.525535    probably_noise mean_revert
          SNACKPACK_RASPBERRY     SNACKPACK      -0.017045  0.017045  0.150817                0.000000 1084.573863    probably_noise mean_revert
         SNACKPACK_STRAWBERRY     SNACKPACK      -0.013804  0.013804  0.221893                0.333333 1211.434709    probably_noise mean_revert
                    PANEL_2X2         PANEL      -0.009771  0.009771  0.094090                0.000000 2272.791082    probably_noise mean_revert
       TRANSLATOR_ASTRO_BLACK    TRANSLATOR      -0.006723  0.006723  0.295921 

## Cell 12 — AR(5) coefficient profiles for likely-exploitable products

For products with |ρ₁| > 0.3, plot the AR(5) coefficient profile to understand how many lags carry signal.  
Also prints the per-day AR(5) R² and compares to AR(1) R² to quantify gain from extra lags.


In [12]:
likely_prods = list(likely['product'])

if len(likely_prods) > 0:
    n_rows = len(likely_prods)
    fig, axes = plt.subplots(n_rows, 3, figsize=(14, 3.5 * n_rows), squeeze=False)
    fig.suptitle('AR(5) Coefficient Profiles — Likely-Exploitable Products', fontsize=12)
    
    rho_cols_ar5 = [f'ar5_rho{k}' for k in range(1, 6)]
    
    for i, prod in enumerate(likely_prods):
        for j, day in enumerate(DAYS):
            ax = axes[i][j]
            row = ar_df[(ar_df['product'] == prod) & (ar_df['day'] == day)]
            if len(row) == 0:
                ax.set_visible(False)
                continue
            row = row.iloc[0]
            rhos = [row[c] for c in rho_cols_ar5]
            lags = range(1, 6)
            colors_bar = ['tomato' if r > 0 else 'steelblue' for r in rhos]
            ax.bar(lags, rhos, color=colors_bar)
            ax.axhline(0, color='black', linewidth=0.8)
            ax.axhline(0.3, color='gray', linestyle='--', linewidth=0.6)
            ax.axhline(-0.3, color='gray', linestyle='--', linewidth=0.6)
            ax.set_xticks(list(lags))
            ax.set_xlabel('Lag', fontsize=7)
            ax.set_ylabel('ρ', fontsize=7)
            ar1_r2 = row['ar1_r2'] if hasattr(row, 'ar1_r2') else ar_df[(ar_df.product==prod)&(ar_df.day==day)]['ar1_r2'].values[0]
            ar5_r2 = row['ar5_r2']
            ax.set_title(f'{prod.split("_",2)[-1]}\nDay {day} | AR1_R²={ar1_r2:.4f} AR5_R²={ar5_r2:.4f}', fontsize=7)
            ax.tick_params(labelsize=6)
    
    plt.tight_layout()
    plt.savefig('/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots/03_ar5_profiles.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("AR(5) profiles saved.")
else:
    print("No likely-exploitable products found under AR(1) rubric; skipping AR(5) profile plot.")

# Print per-day AR(1) vs AR(5) R² for likely + tradable
print("\n=== AR(1) vs AR(5) R² for focus products ===")
focus = ar_df[ar_df['product'].isin(focus_products)][['product','day','ar1_r2','ar5_r2']].copy()
focus['ar5_gain'] = focus['ar5_r2'] - focus['ar1_r2']
print(focus.sort_values(['product','day']).to_string(index=False))


No likely-exploitable products found under AR(1) rubric; skipping AR(5) profile plot.

=== AR(1) vs AR(5) R² for focus products ===
                    product  day   ar1_r2   ar5_r2  ar5_gain
OXYGEN_SHAKE_EVENING_BREATH    2 0.026577 0.027745  0.001168
OXYGEN_SHAKE_EVENING_BREATH    3 0.008999 0.009487  0.000488
OXYGEN_SHAKE_EVENING_BREATH    4 0.005999 0.006176  0.000176
              ROBOT_IRONING    2 0.024204 0.027458  0.003254
              ROBOT_IRONING    3 0.006426 0.007411  0.000984
              ROBOT_IRONING    4 0.013104 0.013282  0.000178


## Cell 13 — Category-level summary

Group the triage counts by category to surface which categories dominate each bucket.


In [13]:
cat_triage = combined.groupby(['category','triage_03']).size().unstack(fill_value=0)
cat_triage['total'] = cat_triage.sum(axis=1)

# Also compute mean |rho| per category
cat_rho = combined.groupby('category').agg(
    mean_abs_rho1=('abs_rho1', 'mean'),
    max_abs_rho1=('abs_rho1', 'max'),
    mean_trend_r2=('trend_r2', 'mean'),
    mean_pnr=('mean_pnr', 'mean'),
).round(4)

print("=== Triage counts per category ===")
print(cat_triage.to_string())
print("\n=== Category-level AR(1) and trend summary ===")
print(cat_rho.sort_values('mean_abs_rho1', ascending=False).to_string())

print("\n=== Top category by mean |ρ₁| ===")
top_cat = cat_rho['mean_abs_rho1'].idxmax()
print(f"  {top_cat}: mean |ρ₁| = {cat_rho.loc[top_cat,'mean_abs_rho1']:.4f}")

print("\n=== Top category by max |ρ₁| ===")
top_cat_max = cat_rho['max_abs_rho1'].idxmax()
print(f"  {top_cat_max}: max |ρ₁| = {cat_rho.loc[top_cat_max,'max_abs_rho1']:.4f}")


=== Triage counts per category ===
triage_03      probably_noise  probably_tradable  total
category                                               
GALAXY_SOUNDS               1                  4      5
MICROCHIP                   0                  5      5
OXYGEN_SHAKE                1                  4      5
PANEL                       1                  4      5
PEBBLES                     0                  5      5
ROBOT                       1                  4      5
SLEEP_POD                   0                  5      5
SNACKPACK                   4                  1      5
TRANSLATOR                  1                  4      5
UV_VISOR                    2                  3      5

=== Category-level AR(1) and trend summary ===
               mean_abs_rho1  max_abs_rho1  mean_trend_r2   mean_pnr
category                                                            
ROBOT                 0.0481        0.1168         0.3696  1347.6315
OXYGEN_SHAKE          0.0399        0.

## Cell 14 — ACF plots for top-5 products by |AR(1) ρ₁|

Autocorrelation function (ACF) up to 50 lags to verify decay pattern:  
- Gradual decay = strong momentum / slow mean-reversion.  
- Sharp cutoff at lag 1 = pure AR(1) process.  
- Oscillating = alternating sign = mean-reversion.  
Plotted for all 3 days to assess stability.


In [14]:
top5_ar1 = ar_summary.head(5)['product'].tolist()
NLAGS = 50

fig, axes = plt.subplots(5, 3, figsize=(15, 15))
fig.suptitle('ACF of Returns — Top-5 by |AR(1) ρ₁| (Days 2/3/4)', fontsize=12)

conf_bound = 1.96 / np.sqrt(10000)  # approx 95% CI band for ~10k observations

for i, prod in enumerate(top5_ar1):
    for j, day in enumerate(DAYS):
        ax = axes[i][j]
        sub = raw[(raw['product'] == prod) & (raw['day'] == day)].sort_values('timestamp')
        levels = sub['mid_price'].values
        returns = np.diff(np.log(levels + 1e-9))
        returns = returns[np.isfinite(returns)]
        
        try:
            acf_vals = acf(returns, nlags=NLAGS, fft=True)
        except:
            ax.set_visible(False)
            continue
        
        lags = range(len(acf_vals))
        ax.bar(lags, acf_vals, width=0.5, color='steelblue', alpha=0.7)
        ax.axhline(conf_bound, color='red', linestyle='--', linewidth=0.8, label='95% CI')
        ax.axhline(-conf_bound, color='red', linestyle='--', linewidth=0.8)
        ax.axhline(0, color='black', linewidth=0.5)
        
        rho_val = ar_df[(ar_df.product==prod)&(ar_df.day==day)]['ar1_rho1'].values
        rho_str = f"ρ₁={rho_val[0]:.3f}" if len(rho_val) > 0 else ""
        ax.set_title(f'{prod.split("_",2)[-1]}\nDay {day} | {rho_str}', fontsize=7)
        ax.tick_params(labelsize=6)
        ax.set_xlabel('Lag', fontsize=6)
        ax.set_ylabel('ACF', fontsize=6)
        ax.set_ylim(-0.6, 0.6)
        if i == 0 and j == 0:
            ax.legend(fontsize=6)

plt.tight_layout()
plt.savefig('/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots/03_acf_top5.png', dpi=100, bbox_inches='tight')
plt.show()
print("ACF plots saved.")


ACF plots saved.


## Cell 15 — Returns distribution sanity check for top AR(1) products

Histogram of returns for the top AR(1) products to check: are returns normal, fat-tailed, or discrete (lattice)?  
Discrete returns (only a few distinct values) could indicate a price-level structure better captured by notebook 01 (distributional lens).


In [15]:
top8_prods = ar_summary.head(8)['product'].tolist()

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
fig.suptitle('Returns distributions — Top-8 by |AR(1) ρ₁| (all days combined)', fontsize=11)

for ax, prod in zip(axes.flatten(), top8_prods):
    all_returns = []
    for day in DAYS:
        sub = raw[(raw['product'] == prod) & (raw['day'] == day)].sort_values('timestamp')
        rets = np.diff(np.log(sub['mid_price'].values + 1e-9))
        all_returns.extend(rets[np.isfinite(rets)])
    
    all_returns = np.array(all_returns)
    n_distinct = len(np.unique(np.round(all_returns, 8)))
    
    ax.hist(all_returns, bins=80, color='steelblue', alpha=0.8, edgecolor='none')
    rho_val = ar_summary[ar_summary['product'] == prod]['mean_ar1_rho1'].values[0]
    ax.set_title(f'{prod.replace("_"," ")}\nρ₁={rho_val:.3f} | n_distinct={n_distinct}', fontsize=7)
    ax.tick_params(labelsize=6)
    ax.set_xlabel('log-return', fontsize=6)

plt.tight_layout()
plt.savefig('/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots/03_returns_dist_top8.png', dpi=100, bbox_inches='tight')
plt.show()
print("Returns distribution plots saved.")

# Print distinct return count for all 50 products
print("\n=== Distinct log-return values (all days) — sanity for lattice structure ===")
dist_rows = []
for prod in PRODUCTS:
    all_r = []
    for day in DAYS:
        sub = raw[(raw['product'] == prod) & (raw['day'] == day)].sort_values('timestamp')
        rets = np.diff(np.log(sub['mid_price'].values + 1e-9))
        all_r.extend(rets[np.isfinite(rets)])
    dist_rows.append({'product': prod, 'n_distinct_returns': len(np.unique(np.round(np.array(all_r), 8)))})
dist_df = pd.DataFrame(dist_rows).sort_values('n_distinct_returns')
print(dist_df.to_string(index=False))


Returns distribution plots saved.

=== Distinct log-return values (all days) — sanity for lattice structure ===


                      product  n_distinct_returns
  OXYGEN_SHAKE_EVENING_BREATH                2058
                ROBOT_IRONING                2472
       OXYGEN_SHAKE_CHOCOLATE               17785
          SNACKPACK_PISTACHIO               17894
                 ROBOT_DISHES               18463
            SNACKPACK_VANILLA               18998
          SNACKPACK_CHOCOLATE               19917
          SNACKPACK_RASPBERRY               20366
     TRANSLATOR_GRAPHITE_MIST               23770
         SNACKPACK_STRAWBERRY               23948
    GALAXY_SOUNDS_DARK_MATTER               24538
  TRANSLATOR_ECLIPSE_CHARCOAL               24574
       TRANSLATOR_ASTRO_BLACK               24623
             MICROCHIP_CIRCLE               24746
                    PANEL_2X2               25208
            OXYGEN_SHAKE_MINT               25305
                    PANEL_4X4               25322
              UV_VISOR_YELLOW               25419
         TRANSLATOR_VOID_BLUE               25435
